# 8장 Astropy 실습: FITS 표시와 좌표 변환

『대중천문학개론』 8장 3.2절(FITS 시각화, 대체 자료 상자)과 3.3절(좌표 변환) 코드 전체.
`OFFLINE = True`이면 모의 영상과 M31 좌표 직접 입력으로 인터넷 없이 돈다. 실제 FITS 파일과 SIMBAD 조회를 쓰려면 `False`로 바꾸고 `m51_r.fits`를 이 폴더에 둔다.

In [ ]:
OFFLINE = True   # 인터넷과 m51_r.fits가 있으면 False로 바꾼다

In [ ]:
# Colab에서 열었으면 저장소를 받아 ch08 폴더로 옮긴다(로컬에서는 아무 일도 하지 않는다)
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("README.md"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Park-Jiwon-Maker/public-astronomy-textbook.git", "repo"], check=True)
    os.chdir("repo/ch08")
print(os.getcwd())

## 대체 자료: 파일이 없을 때 쓰는 모의 영상 (3.2절 상자)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)            # 시드 고정: 누구나 같은 영상
ny, nx = 256, 256
y, x = np.mgrid[:ny, :nx]
data = rng.normal(100.0, 5.0, (ny, nx))   # 하늘 배경 + 잡음
for cx, cy, amp in rng.uniform([20, 20, 200], [236, 236, 3000], (40, 3)):
    # 가우스 별상 40개를 얹는다
    data += amp * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * 2.2**2))
header = {"OBJECT": "synthetic (교육용 모의 영상)", "EXPTIME": 60}
print(data.sum())              # 표시를 바꿔도 이 합은 같아야 한다

In [ ]:
# 오프라인이면 위의 모의 영상을 synthetic.fits로 저장하고, 아래 코드가 m51_r.fits 대신 그 파일을 열게 한다
if OFFLINE:
    from astropy.io import fits as _fits
    _fits.PrimaryHDU(data, header=_fits.Header({"OBJECT": "synthetic (educational mock image)", "EXPTIME": 60})).writeto("synthetic.fits", overwrite=True)
    _open = getattr(_fits, "_book_open", _fits.open)
    _fits._book_open = _open
    _fits.open = lambda name, *a, **k: _open("synthetic.fits" if name == "m51_r.fits" else name, *a, **k)

## 20줄로 하는 FITS 시각화 (3.2절)

In [ ]:
# FITS를 읽고, 헤더를 확인하고, 보기 좋게 표시한다
# 예제 파일 m51_r.fits: 공개 아카이브나 소속 관측소에서 받은 R 밴드 영상. 파일명·출처·
# 내려받은 날짜·전처리(바이어스/플랫 여부)를 관측일지에 적어 둔다. 파일이 없으면 아래 상자의 모의 영상을 쓴다.
import os
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize

path = "m51_r.fits"

with fits.open(path) as hdul:
    hdul.info()                       # 어떤 확장(HDU)이 있는지 목록
    # 영상은 기본 HDU가 아니라 확장 HDU에 있을 수도 있다: 2차원 데이터를 가진 첫 HDU를 고른다
    hdu    = next(h for h in hdul if h.data is not None and h.data.ndim == 2)
    data   = hdu.data                 # 2차원 배열 = 픽셀별 계수
    header = hdu.header

for key in ("OBJECT", "DATE-OBS", "EXPTIME", "FILTER", "TELESCOP"):
    print(f"{key:10s}", header.get(key, "— 없음"))

# zscale 로 표시 구간을 잡고, asinh 로 어두운 구조를 끌어올린다
norm = ImageNormalize(data,
                      interval=ZScaleInterval(),
                      stretch=AsinhStretch())

plt.figure(figsize=(6, 6))
plt.imshow(data, origin="lower", cmap="gray", norm=norm)
plt.colorbar(label="ADU")
plt.title(header.get("OBJECT", "unknown"))
os.makedirs("fig", exist_ok=True)   # 저장 폴더가 없으면 만든다
plt.savefig("fig/m51_r_asinh.png", dpi=150, bbox_inches="tight")

In [ ]:
print("표시 후 합계", data.sum())   # norm은 표시만 바꾸므로 모의 영상의 합계와 같다

## 좌표 변환으로 짜는 관측 계획 (3.3절)

In [ ]:
if OFFLINE:
    import warnings
    from astropy.utils import iers
    from astropy.coordinates import SkyCoord as _SkyCoord
    iers.conf.auto_download = False
    warnings.filterwarnings("ignore", module="astropy.utils.iers")
    _SkyCoord.from_name = classmethod(lambda cls, name, **kw: cls("00h42m44.33s", "+41d16m07.5s", frame="icrs"))   # M31, SIMBAD 기본 좌표

In [ ]:
# 적도좌표 → 지평좌표: "오늘 밤 이 대상은 몇 시에 가장 높이 뜨는가"
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz

# 관측지: 충북대학교 (위도 36.63°N, 경도 127.46°E, 고도 87 m)
site   = EarthLocation(lat=36.63*u.deg, lon=127.46*u.deg, height=87*u.m)
target = SkyCoord.from_name("M31")          # 이름으로 좌표 조회(SIMBAD)

# 한국시간 20시부터 12시간을 10분 간격으로 (UTC = KST − 9h)
t0    = Time("2026-10-15 11:00:00")          # = KST 20:00
times = t0 + np.arange(0, 12, 1/6) * u.hour

altaz = target.transform_to(AltAz(obstime=times, location=site))

best = np.argmax(altaz.alt)
print("최고 고도", f"{altaz.alt[best].deg:.1f}°",
      "시각(KST)", (times[best] + 9*u.hour).iso)
print("고도 30° 이상인 시간",
      f"{(altaz.alt > 30*u.deg).sum() / 6:.1f} 시간")